# Fine-Tuning for Tool Calling and Structured Output

Render tool calls in two model families and build the loss mask, the parser and a BFCL-style checker by hand. Then LoRA fine-tune Qwen3.5-2B-Base on a T4 and measure what fine-tuning and constrained decoding each change.

LLM Training Lab, module 16. Regime: PEFT. Steps 1 to 6 run on a CPU. Steps 7 to 11 need one T4. The cells below are the lab page's code, in order.

In [ ]:
# Runtime check. Run this cell first. It imports nothing heavy, so it works before the install cell.
import os, platform, shutil, subprocess, sys
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")                       # before torch loads: one GPU for Trainer
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print("Python", platform.python_version())
if sys.version_info < (3, 11):
    print("PROBLEM Python 3.11 or later is needed, 3.12 recommended. Start a current Colab or Kaggle session.")
free = shutil.disk_usage(".").free / 1e9
print("Free disk", round(free, 1), "GB")
if free < 12:
    print("PROBLEM the models need about 9 GB of downloads (Qwen3.5-2B-Base, Qwen3.5-0.8B-Base, Qwen3-1.7B-Base). Free some disk.")
try:   # a CPU-only runtime has no nvidia-smi at all
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    gpu = ""
print("GPU", gpu or "none")
if not gpu:
    print("Steps 1 to 6 run on this CPU. Steps 7 to 11 need a GPU: Colab, Runtime > Change runtime type > T4 GPU. Kaggle, Settings > Accelerator > GPU T4 x2.")
elif "P100" in gpu:
    print("PROBLEM a P100 has no kernels in current PyTorch. Pick a T4 instead.")
from importlib.metadata import version, PackageNotFoundError
for pkg, want in [("transformers", "5.22.1"), ("trl", "1.17.1"), ("peft", "0.21.2"), ("accelerate", "1.15.0"), ("datasets", "5.1.0"), ("bfcl-eval", "2026.3.23")]:
    try:
        have = version(pkg)
    except PackageNotFoundError:
        have = "not installed"
    print(pkg, have, "" if have == want else "(the next cell installs " + want + ")")
print("Runtime OK" if sys.version_info >= (3, 11) else "Fix the PROBLEM lines, then run this cell again.")

In [ ]:
# Pinned versions of our recorded runs. torch and numpy stay as the runtime ships them.
# torchao is removed because PEFT 0.21.2 refuses the version on the Kaggle image.
# bfcl-eval is installed without its dependencies: they pin numpy 1.26.4, which would break the runtime's torch.
# If transformers was already imported in this session, restart the session after this cell, then run from step 1.
%pip uninstall -y -q torchao
%pip install -q "transformers==5.22.1" "trl==1.17.1" "peft==0.21.2" "accelerate==1.15.0" "datasets==5.1.0" "tokenizers==0.23.3" "huggingface_hub==2.4.0"
%pip install -q --no-deps "bfcl-eval==2026.3.23"

The next cell writes `tool_common.py`, the helpers the steps import as `C`: the data selection, the BFCL loader, the parsers, the checker, the constrained decoder and the generation loop. Steps 2 to 8 show the functions you build yourself.

In [ ]:
%%writefile tool_common.py
# tool_common.py: the shared code of the lltToolUse lab (data, labels, parsers, checker, constrained decoding,
# evaluation). The lab page shows these functions in its steps; the notebook writes this file in its third cell.
# No backslashes anywhere in this file: the page keeps its code in JS template literals. NL is the newline.
import json, random, re

NL = chr(10)

# ---------- Step 2. Hermes rows to structured messages ----------
def hermes_to_chat(row):
    # The Hermes text writes calls as JSON inside <tool_call> tags. Turn them into structured tool_calls,
    # so the model's own chat template renders them in the model's own format.
    tools = json.loads(row["tools"]) if isinstance(row["tools"], str) else row["tools"]
    messages = []
    for turn in row["conversations"]:
        role, text = turn["from"], turn["value"]
        if role == "system":
            continue                                   # the template writes its own tool prompt from the tools list
        if role == "human":
            messages.append({"role": "user", "content": text})
        elif role == "tool":
            body = text.replace("<tool_response>", "").replace("</tool_response>", "").strip()
            messages.append({"role": "tool", "content": body})
        elif role == "gpt":
            calls = []
            for block in re.findall("<tool_call>(.*?)</tool_call>", text, re.S):
                call = json.loads(block)               # raises on a malformed call; the caller drops the row
                calls.append({"type": "function", "function": {"name": call["name"], "arguments": call["arguments"]}})
            content = re.sub("<tool_call>.*?</tool_call>", "", text, flags=re.S).strip()
            msg = {"role": "assistant", "content": content}
            if calls:
                msg["tool_calls"] = calls
            messages.append(msg)
    return {"messages": messages, "tools": tools or []}


def parses(row):
    try:
        hermes_to_chat(row)
        return True
    except (json.JSONDecodeError, KeyError, TypeError):
        return False


def select_rows(rows, n, render, max_tokens):
    # Keep rows in file order whose calls parse and whose rendered length fits. Returns kept rows and drop counts.
    kept, dropped = [], {"bad_json": 0, "too_long": 0, "no_tools": 0}
    for row in rows:
        try:
            chat = hermes_to_chat(row)
        except (json.JSONDecodeError, KeyError, TypeError):
            dropped["bad_json"] += 1
            continue
        if not chat["tools"]:
            dropped["no_tools"] += 1
            continue
        ids = render(chat)
        if len(ids) > max_tokens:
            dropped["too_long"] += 1
            continue
        kept.append(chat)
        if len(kept) == n:
            break
    return kept, dropped


# ---------- Step 3. The label mask ----------
def build_labels(ids, header, im_end, newline):
    # Loss on each assistant turn: every token after the header up to and including <|im_end|> and the newline
    # after it, which is the span TRL's training template marks with {% generation %}. Everything else is -100:
    # the tool prompt, the user turns and every tool response.
    labels = [-100] * len(ids)
    h = len(header)
    t = 0
    while t < len(ids):
        if ids[t:t + h] == header:
            t += h
            while t < len(ids):
                labels[t] = ids[t]
                t += 1
                if ids[t - 1] == im_end:
                    if t < len(ids) and ids[t] == newline:
                        labels[t] = ids[t]
                        t += 1
                    break
        else:
            t += 1
    return labels


# ---------- Step 4. BFCL items with JSON Schema tools ----------
BFCL_TYPES = {"dict": "object", "float": "number", "tuple": "array", "any": "string", "list": "array", "bool": "boolean"}

def to_json_schema(p):
    # BFCL writes types in its own dialect (dict, float, tuple). Map them to JSON Schema, recursively.
    p = dict(p)
    if "type" in p:
        p["type"] = BFCL_TYPES.get(p["type"], p["type"])
    if "properties" in p:
        p["properties"] = {k: to_json_schema(v) for k, v in p["properties"].items()}
    if "items" in p and isinstance(p["items"], dict):
        p["items"] = to_json_schema(p["items"])
    return p

def bfcl_tools(functions):
    return [{"type": "function", "function": {"name": f["name"], "description": f.get("description", ""),
                                              "parameters": to_json_schema(f["parameters"])}} for f in functions]

def load_bfcl(data_dir, plan):
    # plan: {category: stride}. Takes every stride-th item of BFCL_v4_<category>.json, in file order.
    items = []
    for cat, stride in plan.items():
        qs = [json.loads(l) for l in open(data_dir + "/BFCL_v4_" + cat + ".json", encoding="utf-8") if l.strip()]
        gt = {}
        try:
            for l in open(data_dir + "/possible_answer/BFCL_v4_" + cat + ".json", encoding="utf-8"):
                if l.strip():
                    a = json.loads(l)
                    gt[a["id"]] = a["ground_truth"]
        except FileNotFoundError:
            pass                                       # irrelevance has no answers: the right output is no call
        for q in qs[::stride]:
            items.append({"id": q["id"], "category": cat, "messages": q["question"][0], "functions": q["function"],
                          "tools": bfcl_tools(q["function"]), "ground_truth": gt.get(q["id"])})
    return items


# ---------- Step 5. Parse the model's output, one parser per family ----------
def coerce(raw, typ):
    # Type an XML parameter value with the tool's schema, as transformers' parse_response does.
    try:
        if typ == "integer":
            return int(raw)
        if typ == "number":
            return float(raw) if ("." in raw or "e" in raw.lower()) else int(raw)
        if typ == "boolean" and raw.lower() in ("true", "false"):
            return raw.lower() == "true"
        if typ in ("array", "object"):
            return json.loads(raw)
    except ValueError:
        pass
    return raw

def parse_xml_calls(text, tools):
    # Qwen3.5 format: <tool_call> <function=NAME> <parameter=KEY> VALUE </parameter> ... </function> </tool_call>
    # Returns a list of {name: arguments}; [] when there is no call; None when a call is malformed.
    if text.count("<tool_call>") != text.count("</tool_call>"):
        return None
    props = {t["function"]["name"]: t["function"]["parameters"].get("properties", {}) for t in tools}
    calls = []
    for block in re.findall("<tool_call>(.*?)</tool_call>", text, re.S):
        found = re.search("<function=([^>]+)>", block)
        if found is None or block.count("<parameter=") != block.count("</parameter>"):
            return None
        name = found.group(1).strip()
        args = {}
        for key, raw in re.findall("<parameter=([^>]+)>(.*?)</parameter>", block, re.S):
            schema = props.get(name, {}).get(key.strip(), {})
            args[key.strip()] = coerce(raw.strip(), schema.get("type"))
        calls.append({name: args})
    return calls

def parse_json_calls(text, tools=None):
    # Qwen3 (Hermes) format: <tool_call> {"name": ..., "arguments": {...}} </tool_call>
    if text.count("<tool_call>") != text.count("</tool_call>"):
        return None
    calls = []
    for block in re.findall("<tool_call>(.*?)</tool_call>", text, re.S):
        try:
            call = json.loads(block)
        except json.JSONDecodeError:
            return None
        if not isinstance(call, dict) or "name" not in call or not isinstance(call.get("arguments", {}), dict):
            return None
        calls.append({call["name"]: call.get("arguments", {})})
    return calls


# ---------- Step 6. A BFCL-style AST check ----------
def norm(s):
    # BFCL compares strings without spaces and the punctuation , . / - _ * ^, ignoring case
    for ch in " ,./-_*^":
        s = s.replace(ch, "")
    return s.lower().replace("'", '"')

def value_ok(value, allowed):
    for a in allowed:
        if a is None and value is None:
            return True
        if isinstance(a, str) and isinstance(value, str) and norm(a) == norm(value):
            return True
        if isinstance(a, bool) or isinstance(value, bool):
            if a is value:
                return True
            continue
        if isinstance(a, (int, float)) and isinstance(value, (int, float)) and a == value:
            return True
        if isinstance(a, list) and isinstance(value, list) and len(a) == len(value) and all(value_ok(v, [x]) for v, x in zip(value, a)):
            return True
        if isinstance(a, dict) and isinstance(value, dict) and set(value) <= set(a) and all(value_ok(value[k], a[k]) for k in value) and all("" in a[k] for k in a if k not in value):
            return True
    return False

def call_ok(call, expected, schema):
    # call {name: args}; expected {name: {param: [allowed values, "" if it may be left out]}}
    name, allowed = next(iter(expected.items()))
    if name not in call:
        return False
    args = call[name]
    if any(p not in args for p in schema.get("required", [])):
        return False
    if any(p not in allowed or p not in schema.get("properties", {}) for p in args):
        return False
    if any(p not in args and "" not in allowed[p] for p in allowed):
        return False
    return all(value_ok(v, allowed[p]) for p, v in args.items())

def check(item, calls):
    # True when the parsed calls match the ground truth. calls is None when the reply had a malformed call.
    # As in BFCL, an irrelevance item is right when no well-formed call comes out.
    if item["category"].endswith("irrelevance"):
        return not calls
    if not calls:
        return False
    expected = item["ground_truth"]
    if len(calls) != len(expected):
        return False
    schemas = {f["name"]: f["parameters"] for f in item["functions"]}
    unused = list(range(len(calls)))
    for exp in expected:                                   # any order: give each expected call its own match
        hit = next((i for i in unused if call_ok(calls[i], exp, schemas.get(next(iter(exp)), {}))), None)
        if hit is None:
            return False
        unused.remove(hit)
    return True


# ---------- Step 8. Constrained decoding: force a well-formed call with offered names ----------
TEXT_INDEX = {}

class CallsOnly:
    # A logits processor for the Qwen3.5 format. Each reply must be one or more tool calls. Function and parameter
    # names come from the offered tools; parameter values stay free text. This is tool_choice "required" plus a grammar.
    def __init__(self, tok, prompt_len, tools_per_row):
        self.tok, self.prompt_len, self.rows = tok, prompt_len, tools_per_row
        self.end_id = tok.convert_tokens_to_ids("<|im_end|>")
        if id(tok) not in TEXT_INDEX:                    # token text -> ids, built once per tokenizer
            index = {}
            for i in range(len(tok)):
                s = tok.decode([i])
                if s and s.isascii():
                    index.setdefault(s, []).append(i)
            TEXT_INDEX[id(tok)] = index
        self.by_text = TEXT_INDEX[id(tok)]

    def options(self, text, tools):
        # The strings that may come next. None means free text (inside a parameter value).
        names = {t["function"]["name"]: list(t["function"]["parameters"].get("properties", {})) for t in tools}
        pos = 0
        start = "<tool_call>" + NL + "<function="
        while True:
            rest = text[pos:]
            if not rest.startswith(start):
                return [start[len(rest):]] if start.startswith(rest) else []
            pos += len(start)
            rest = text[pos:]
            heads = [n + ">" + NL for n in names]
            got = [h for h in heads if rest.startswith(h)]
            if not got:
                return [h[len(rest):] for h in heads if h.startswith(rest)]
            name = got[0][:-2]
            pos += len(got[0])
            used = set()
            while True:
                rest = text[pos:]
                opts = ["<parameter=" + p + ">" + NL for p in names[name] if p not in used] + ["</function>" + NL + "</tool_call>"]
                got = [o for o in opts if rest.startswith(o)]
                if not got:
                    return [o[len(rest):] for o in opts if o.startswith(rest)]
                pos += len(got[0])
                if got[0].startswith("</function>"):
                    break
                used.add(got[0][len("<parameter="):-2])
                close = text.find("</parameter>", pos)
                if close < 0:
                    return None                          # inside a value: free text
                pos = close + len("</parameter>")
                rest = text[pos:]
                if not rest.startswith(NL):
                    return [NL] if rest == "" else []
                pos += 1
            rest = text[pos:]
            if rest == "":
                return [NL + start, "<|im_end|>"]
            if not rest.startswith(NL):
                return []
            pos += 1

    def __call__(self, input_ids, scores):
        for r in range(input_ids.shape[0]):
            text = self.tok.decode(input_ids[r, self.prompt_len:], skip_special_tokens=False)
            if "<|im_end|>" in text:
                continue
            opts = self.options(text, self.rows[r])
            if opts is None:
                continue
            allowed = set()
            for o in opts:
                if o == "<|im_end|>":
                    allowed.add(self.end_id)
                    continue
                for k in range(1, len(o) + 1):
                    allowed.update(self.by_text.get(o[:k], []))
            if not allowed:
                allowed.add(self.end_id)                  # a dead end: stop, and the parser reports the bad call
            keep = scores[r, list(allowed)].clone()
            scores[r] = float("-inf")
            scores[r, list(allowed)] = keep
        return scores


# ---------- Step 7. Generate replies for many items ----------
def generate(model, tok, items, batch_size=32, max_new_tokens=384, constrained=False, device="cuda"):
    import torch
    from transformers import LogitsProcessorList
    tok.padding_side = "left"
    stops = [tok.convert_tokens_to_ids("<|im_end|>"), tok.eos_token_id]
    prompts = [tok.apply_chat_template(it["messages"], tools=it["tools"], add_generation_prompt=True, tokenize=False) for it in items]
    order = sorted(range(len(items)), key=lambda i: len(prompts[i]))
    out = [None] * len(items)
    for b in range(0, len(order), batch_size):
        idx = order[b:b + batch_size]
        enc = tok([prompts[i] for i in idx], return_tensors="pt", padding=True, add_special_tokens=False).to(device)
        procs = LogitsProcessorList()
        if constrained:
            procs.append(CallsOnly(tok, enc["input_ids"].shape[1], [items[i]["tools"] for i in idx]))
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, eos_token_id=stops,
                                 pad_token_id=tok.pad_token_id, logits_processor=procs)
        for j, i in enumerate(idx):
            text = tok.decode(gen[j, enc["input_ids"].shape[1]:], skip_special_tokens=False)
            out[i] = text.split("<|im_end|>")[0].replace(tok.pad_token, "")
    return out


# ---------- Scoring ----------
def score(items, outputs, parse):
    rows = []
    for it, text in zip(items, outputs):
        calls = parse(text, it["tools"])
        rows.append({"id": it["id"], "category": it["category"], "parsed": calls is not None,
                     "calls": calls, "correct": check(it, calls)})
    return rows

def summary(rows):
    cats = {}
    for r in rows:
        c = cats.setdefault(r["category"], [0, 0, 0])
        c[0] += 1
        c[1] += int(r["correct"])
        c[2] += int(r["parsed"])
    total = len(rows)
    return {"accuracy": round(sum(r["correct"] for r in rows) / total, 4),
            "parseRate": round(sum(r["parsed"] for r in rows) / total, 4),
            "byCategory": {k: {"n": v[0], "accuracy": round(v[1] / v[0], 4), "parseRate": round(v[2] / v[0], 4)} for k, v in cats.items()}}

def paired_bootstrap(a, b, n=10000, seed=0):
    # a, b: lists of 0/1 per item for two systems on the same items. Returns the mean difference b - a and a 95% CI.
    rng = random.Random(seed)
    d = [y - x for x, y in zip(a, b)]
    means = sorted(sum(d[rng.randrange(len(d))] for _ in d) / len(d) for _ in range(n))
    return round(sum(d) / len(d), 4), round(means[int(0.025 * n)], 4), round(means[int(0.975 * n)], 4)


def official_check():
    # BFCL's own AST checker from the bfcl-eval package (pip install --no-deps bfcl-eval==2026.3.23).
    # Its model_config module imports every API client, so stand in a one-entry table for it.
    import sys, types
    stub = types.ModuleType("bfcl_eval.constants.model_config")
    stub.MODEL_CONFIG_MAPPING = {"lab": types.SimpleNamespace(underscore_to_dot=False)}
    sys.modules["bfcl_eval.constants.model_config"] = stub
    from bfcl_eval.eval_checker.ast_eval.ast_checker import ast_checker
    from bfcl_eval.constants.enums import Language

    def run(item, calls):
        if item["category"].endswith("irrelevance"):
            return not calls
        if not calls:
            return False
        return ast_checker(item["functions"], calls, item["ground_truth"], Language.PYTHON, item["category"], "lab")["valid"]
    return run


# ---------- Step 6. Render a ground-truth answer, to test the parser and the checker ----------
def first_value(v):
    # BFCL nests allowed-value lists inside dict answers: {"width": [20]} means width must be 20.
    if isinstance(v, dict):
        return {k: first_value([x for x in vals if x != ""][0]) for k, vals in v.items() if [x for x in vals if x != ""]}
    if isinstance(v, list):
        return [first_value(x) for x in v]
    return v

def gt_reply(tok, item):
    # The reply the template writes for the first allowed answer. Irrelevance items get an empty reply.
    if item["ground_truth"] is None:
        return ""
    calls = []
    for exp in item["ground_truth"]:
        name, allowed = next(iter(exp.items()))
        args = {p: first_value([x for x in vals if x != ""][0]) for p, vals in allowed.items() if [x for x in vals if x != ""]}
        calls.append({"type": "function", "function": {"name": name, "arguments": args}})
    prompt = tok.apply_chat_template(item["messages"], tools=item["tools"], tokenize=False, add_generation_prompt=True)
    full = tok.apply_chat_template(item["messages"] + [{"role": "assistant", "content": "", "tool_calls": calls}],
                                   tools=item["tools"], tokenize=False)
    return full[len(prompt):].split("<|im_end|>")[0]


# ---------- Step 8. LoRA on a frozen fp16 base with TRL ----------
def train_lora(name, tok, rows, seed=0, rank=16, lr=2e-4, out="lora-out", special=None):
    import torch
    from datasets import Dataset
    from peft import LoraConfig, get_peft_model
    from transformers import AutoModelForCausalLM
    from trl import SFTConfig, SFTTrainer
    model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16).to("cuda")   # frozen base, fp16
    model.enable_input_require_grads()
    if special:                                          # step 10: give untrained special tokens distinct rows
        E = model.get_input_embeddings().weight
        with torch.no_grad():
            for s in special:                            # the mean of the rows of its spelling, "<" + "tool_call>"
                pieces = tok.encode(s[0], add_special_tokens=False) + tok.encode(s[1:], add_special_tokens=False)
                E[tok.convert_tokens_to_ids(s)] = E[pieces].mean(0)
    model = get_peft_model(model, LoraConfig(r=rank, lora_alpha=2 * rank, lora_dropout=0.0,
                                             target_modules="all-linear", task_type="CAUSAL_LM"))
    args = SFTConfig(output_dir=out, report_to="none", save_strategy="no", seed=seed, data_seed=seed,
                     fp16=True, bf16=False,              # a T4 has no bf16 tensor cores; SFTConfig defaults to bf16
                     learning_rate=lr, lr_scheduler_type="cosine", warmup_steps=10, num_train_epochs=1,
                     per_device_train_batch_size=2, gradient_accumulation_steps=2, max_length=1536,
                     gradient_checkpointing=True, logging_steps=5)
    trainer = SFTTrainer(model=model, args=args, processing_class=tok, train_dataset=Dataset.from_list(rows))
    for p in trainer.model.parameters():                 # GradScaler needs fp32 adapters; re-cast any that are not
        if p.requires_grad and p.dtype != torch.float32:
            p.data = p.data.float()
    trainer.train()
    return trainer


# ---------- Step 11. Conversations whose tools cannot answer ----------
NO_TOOL_REPLY = "None of the functions I have can do this, so I will not call one. I can answer in plain text if that helps."

def make_negatives(rows, start, count, shift):
    # Pair each request with the tools of another conversation that share no function name with its own tools.
    # The target reply is plain text with no call: what BFCL's irrelevance items ask for.
    out = []
    for k in range(start, start + count):
        a, b = rows[k], rows[(k - shift) % len(rows)]
        own = {t["function"]["name"] for t in a["tools"]}
        other = {t["function"]["name"] for t in b["tools"]}
        if own & other:
            continue
        out.append({"messages": [a["messages"][0], {"role": "assistant", "content": NO_TOOL_REPLY}], "tools": b["tools"]})
    return out

## Step 1. Render a tool call in two model families

What you should see. For Qwen3.5-2B-Base the tool prompt adds 275 tokens and the generation prompt ends with an empty `<think>` block. The reply is `<tool_call>`, `<function=get_weather>` and three parameter blocks, with the boolean written as `True`. For Qwen3-1.7B-Base the tool prompt adds 156 tokens, and the reply is one line of JSON, `{"name": "get_weather", "arguments": {"city": "Pune", "days": 3, "metric": true}}`, after an empty `<think>` block.

In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")     # Kaggle has 2 GPUs, and Trainer refuses a model split over both
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import gc, json, re, time, torch
from transformers import AutoTokenizer
import tool_common as C        # written by the notebook's third cell; also labs/lltToolUse/tool_common.py in the repo

MAIN, REF = "Qwen/Qwen3.5-2B-Base", "Qwen/Qwen3-1.7B-Base"
tok, ref_tok = AutoTokenizer.from_pretrained(MAIN), AutoTokenizer.from_pretrained(REF)

weather = {"type": "function", "function": {"name": "get_weather", "description": "Get the weather for a city.",
           "parameters": {"type": "object", "required": ["city"], "properties": {
               "city": {"type": "string"}, "days": {"type": "integer"}, "metric": {"type": "boolean"}}}}}
ask = [{"role": "user", "content": "Weather in Pune for 3 days?"}]
call = {"role": "assistant", "content": "", "tool_calls": [{"type": "function", "function": {
    "name": "get_weather", "arguments": {"city": "Pune", "days": 3, "metric": True}}}]}

for name, t in [(MAIN, tok), (REF, ref_tok)]:
    prompt = t.apply_chat_template(ask, tools=[weather], tokenize=False, add_generation_prompt=True)
    bare = t.apply_chat_template(ask, tokenize=False, add_generation_prompt=True)
    full = t.apply_chat_template(ask + [call], tools=[weather], tokenize=False)
    extra = len(t(prompt)["input_ids"]) - len(t(bare)["input_ids"])
    print("==", name, "| the tool prompt adds", extra, "tokens | the prompt ends", repr(prompt[-30:]))
    print(full[len(prompt):])

## Step 2. Turn the Hermes data into structured tool calls

What you should see. The Hermes reply prints as JSON text inside `<tool_call>` tags, and the converted call prints as a dict with `name` and `arguments`. The single-turn file keeps 600 rows after reading 630, dropping 30 that render to more than 1,536 tokens. The glaive file keeps its first 600. The last line reports 1,200 training conversations and 793 of 1,893 single-turn rows with invalid JSON (lltToolUse-t1-data).

In [ ]:
from huggingface_hub import hf_hub_download
from trl.chat_template_utils import get_training_chat_template

def hermes(file):
    path = hf_hub_download("NousResearch/hermes-function-calling-v1", file, repo_type="dataset")
    return json.load(open(path, encoding="utf-8"))

single, glaive = hermes("func-calling-singleturn.json"), hermes("glaive-function-calling-5k.json")
print(single[0]["conversations"][2]["value"][:150])       # a Hermes reply: the call is JSON text

def hermes_to_chat(row):
    # The Hermes text writes calls as JSON inside <tool_call> tags. Turn them into structured tool_calls,
    # so the model's own chat template renders them in the model's own format.
    tools = json.loads(row["tools"]) if isinstance(row["tools"], str) else row["tools"]
    messages = []
    for turn in row["conversations"]:
        role, text = turn["from"], turn["value"]
        if role == "system":
            continue                                   # the template writes its own tool prompt from the tools list
        if role == "human":
            messages.append({"role": "user", "content": text})
        elif role == "tool":
            body = text.replace("<tool_response>", "").replace("</tool_response>", "").strip()
            messages.append({"role": "tool", "content": body})
        elif role == "gpt":
            calls = []
            for block in re.findall("<tool_call>(.*?)</tool_call>", text, re.S):
                call = json.loads(block)               # raises on a malformed call; the caller drops the row
                calls.append({"type": "function", "function": {"name": call["name"], "arguments": call["arguments"]}})
            content = re.sub("<tool_call>.*?</tool_call>", "", text, flags=re.S).strip()
            msg = {"role": "assistant", "content": content}
            if calls:
                msg["tool_calls"] = calls
            messages.append(msg)
    return {"messages": messages, "tools": tools or []}

print(hermes_to_chat(single[0])["messages"][1]["tool_calls"][0])

train_tpl = get_training_chat_template(tok)        # Qwen3.5's template with generation markers, shipped in TRL
def render(chat):
    return list(tok.apply_chat_template(chat["messages"], tools=chat["tools"], chat_template=train_tpl)["input_ids"])

N_PER_SOURCE = 600
train = []
for name, rows in [("singleturn", single), ("glaive", glaive)]:
    kept, dropped = C.select_rows(rows, N_PER_SOURCE, render, max_tokens=1536)
    train += kept
    print(name, "| kept", len(kept), "| dropped", dropped)
bad = sum(1 for row in single if not C.parses(row))
print(len(train), "training conversations | singleturn rows with a call that is not valid JSON:", bad, "of", len(single))

## Step 3. Mask everything except the assistant turns

What you should see. The printout shows the user turn and the tool response as plain text, and every assistant token in brackets, from `[<think>]` to `[<|im_end|>]` and the newline after it. The shares with loss are 0.141 for single-turn and 0.229 for glaive. The check prints arguments padded with None after `from_list`, `labels equal: True` on a batch of [4, 1427], and a loss difference below 2e-07. Your loss and TRL's were 4.4e-08 apart on our laptop CPU (lltToolUse-t1-maskloss) and 1.8e-07 apart on a Kaggle T4, both at 0.46009 (lltToolUse-T2-notebook-q35-2b-s0).

In [ ]:
def build_labels(ids, header, im_end, newline):
    # Loss on each assistant turn: every token after the header up to and including <|im_end|> and the newline
    # after it, which is the span TRL's training template marks with {% generation %}. Everything else is -100:
    # the tool prompt, the user turns and every tool response.
    labels = [-100] * len(ids)
    h = len(header)
    t = 0
    while t < len(ids):
        if ids[t:t + h] == header:
            t += h
            while t < len(ids):
                labels[t] = ids[t]
                t += 1
                if ids[t - 1] == im_end:
                    if t < len(ids) and ids[t] == newline:
                        labels[t] = ids[t]
                        t += 1
                    break
        else:
            t += 1
    return labels

HEADER = tok.encode("<|im_start|>assistant" + C.NL, add_special_tokens=False)
IM_END, NEWLINE = tok.convert_tokens_to_ids("<|im_end|>"), tok.encode(C.NL, add_special_tokens=False)[0]
ids = render(train[N_PER_SOURCE])                  # the first glaive chat: a call, a tool response, then an answer
labels = build_labels(ids, HEADER, IM_END, NEWLINE)
shown = "".join("[" + tok.decode([i]) + "]" if l != -100 else tok.decode([i]) for i, l in zip(ids, labels))
print(shown[shown.index("<|im_start|>user"):][:900])

rows = []
for c in train:
    ids = render(c)
    rows.append({"input_ids": ids, "labels": build_labels(ids, HEADER, IM_END, NEWLINE)})
for name, part in [("singleturn", rows[:N_PER_SOURCE]), ("glaive", rows[N_PER_SOURCE:])]:
    n_tok = sum(len(r["input_ids"]) for r in part)
    n_loss = sum(sum(l != -100 for l in r["labels"]) for r in part)
    print(name, "| tokens", n_tok, "| share with loss", round(n_loss / n_tok, 3))

In [ ]:
from datasets import Dataset, Features, Json, List
from transformers import AutoModelForCausalLM
from trl import SFTConfig, SFTTrainer

eight = train[:4] + train[N_PER_SOURCE:N_PER_SOURCE + 4]
as_rows = [{"messages": c["messages"], "tools": json.dumps(c["tools"])} for c in eight]
plain = Dataset.from_list(as_rows)                 # Arrow merges every call's argument keys into one struct
print("arguments of the first call after from_list:", plain[0]["messages"][1]["tool_calls"][0]["function"]["arguments"])
typed = Dataset.from_list(as_rows, features=Features({"messages": List(Json()), "tools": Json()}))

dev = "cuda" if torch.cuda.is_available() else "cpu"
small = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3.5-0.8B-Base", dtype=torch.float32).to(dev)
args = SFTConfig(output_dir="check", report_to="none", assistant_only_loss=True, max_length=2048,
                 per_device_train_batch_size=4, bf16=False, use_cpu=(dev == "cpu"))
trainer = SFTTrainer(model=small, args=args, processing_class=tok, train_dataset=typed)
batch = {k: v.to(dev) for k, v in next(iter(trainer.get_train_dataloader())).items()}

ours = torch.full_like(batch["input_ids"], -100)
for r in range(len(ours)):
    n = int(batch["attention_mask"][r].sum())
    ours[r, :n] = torch.tensor(build_labels(batch["input_ids"][r, :n].tolist(), HEADER, IM_END, NEWLINE))
labels_equal = torch.equal(ours, batch["labels"])
print("labels equal:", labels_equal, "| batch", list(batch["input_ids"].shape))

with torch.no_grad():
    total, count = 0.0, 0
    for r in range(len(ours)):                      # one row at a time: a 248k-wide fp32 logit row is about 1 MB per token
        logits = small(input_ids=batch["input_ids"][r:r + 1], attention_mask=batch["attention_mask"][r:r + 1]).logits[0]
        keep = ours[r, 1:] != -100
        total += torch.nn.functional.cross_entropy(logits[:-1][keep].double(), ours[r, 1:][keep], reduction="sum").item()
        count += int(keep.sum())
    lib = trainer.compute_loss(trainer.model, batch).item()
print("own", total / count, "| TRL", lib, "| difference", abs(total / count - lib))
del small, trainer
gc.collect()
torch.cuda.empty_cache()

## Step 4. Load a BFCL V4 subset with JSON Schema tools

What you should see. 310 items, 100 simple, 50 multiple, 50 parallel, 50 parallel multiple and 60 irrelevance. The first is the triangle question, `base` has BFCL type `integer`, the parameters object maps from `dict` to `object`, and the answer allows `unit` to be "units" or left out. 9 of the subset's 377 function names also appear in the training data, all generic, such as `calculate_bmi` and `get_stock_price` (lltToolUse-t1-data).

In [ ]:
import bfcl_eval                                     # pip install --no-deps bfcl-eval==2026.3.23, in the install cell
BFCL = os.path.join(os.path.dirname(bfcl_eval.__file__), "data")
BFCL_TYPES = {"dict": "object", "float": "number", "tuple": "array", "any": "string", "list": "array", "bool": "boolean"}

def to_json_schema(p):
    # BFCL writes types in its own dialect (dict, float, tuple). Map them to JSON Schema, recursively.
    p = dict(p)
    if "type" in p:
        p["type"] = BFCL_TYPES.get(p["type"], p["type"])
    if "properties" in p:
        p["properties"] = {k: to_json_schema(v) for k, v in p["properties"].items()}
    if "items" in p and isinstance(p["items"], dict):
        p["items"] = to_json_schema(p["items"])
    return p

PLAN = {"simple_python": 4, "multiple": 4, "parallel": 4, "parallel_multiple": 4, "irrelevance": 4}   # every 4th item
items = C.load_bfcl(BFCL, PLAN)
print(len(items), "items", {k: sum(it["category"] == k for it in items) for k in PLAN})
it = items[0]
print(it["messages"][0]["content"])
print(it["functions"][0]["parameters"]["properties"]["base"], "->", it["tools"][0]["function"]["parameters"]["type"])
print("answer:", it["ground_truth"])

## Step 5. Parse each family's calls with its own parser

What you should see. The right answer to the first parallel item prints in Qwen3.5 format with one `<parameter>` block per argument. The XML parser returns the call with ints and booleans typed, and the JSON parser returns None. The Qwen3 version of the same answer prints as JSON after an empty `<think>` block.

In [ ]:
def coerce(raw, typ):
    # Type an XML parameter value with the tool's schema, as transformers' parse_response does.
    try:
        if typ == "integer":
            return int(raw)
        if typ == "number":
            return float(raw) if ("." in raw or "e" in raw.lower()) else int(raw)
        if typ == "boolean" and raw.lower() in ("true", "false"):
            return raw.lower() == "true"
        if typ in ("array", "object"):
            return json.loads(raw)
    except ValueError:
        pass
    return raw

def parse_xml_calls(text, tools):
    # Qwen3.5 format: <tool_call> <function=NAME> <parameter=KEY> VALUE </parameter> ... </function> </tool_call>
    # Returns a list of {name: arguments}; [] when there is no call; None when a call is malformed.
    if text.count("<tool_call>") != text.count("</tool_call>"):
        return None
    props = {t["function"]["name"]: t["function"]["parameters"].get("properties", {}) for t in tools}
    calls = []
    for block in re.findall("<tool_call>(.*?)</tool_call>", text, re.S):
        found = re.search("<function=([^>]+)>", block)
        if found is None or block.count("<parameter=") != block.count("</parameter>"):
            return None
        name = found.group(1).strip()
        args = {}
        for key, raw in re.findall("<parameter=([^>]+)>(.*?)</parameter>", block, re.S):
            schema = props.get(name, {}).get(key.strip(), {})
            args[key.strip()] = coerce(raw.strip(), schema.get("type"))
        calls.append({name: args})
    return calls

par = next(it for it in items if it["category"] == "parallel")
reply = C.gt_reply(tok, par)                       # the right answer to the first parallel item, in Qwen3.5 format
print(reply)
print("XML parser :", parse_xml_calls(reply, par["tools"]))
print("JSON parser:", C.parse_json_calls(reply, par["tools"]))
print("Qwen3 reply:", C.gt_reply(ref_tok, par)[:120])

## Step 6. Score a call the way BFCL does

What you should see. Both formats score their right answers 308 or 309 of 310 correct, and your check agrees with BFCL's on all 310 items each time (lltToolUse-t1-roundtrip). Qwen3.5 misses two. `parallel_152` allows a null `mod`, which the XML format can only write as the text None. `parallel_multiple_12` lists an answer parameter the schema does not have. Qwen3 misses only the second.

In [ ]:
def call_ok(call, expected, schema):
    # call {name: args}; expected {name: {param: [allowed values, "" if it may be left out]}}
    name, allowed = next(iter(expected.items()))
    if name not in call:
        return False
    args = call[name]
    if any(p not in args for p in schema.get("required", [])):
        return False
    if any(p not in allowed or p not in schema.get("properties", {}) for p in args):
        return False
    if any(p not in args and "" not in allowed[p] for p in allowed):
        return False
    return all(C.value_ok(v, allowed[p]) for p, v in args.items())

def check(item, calls):
    # True when the parsed calls match the ground truth. calls is None when the reply had a malformed call.
    # As in BFCL, an irrelevance item is right when no well-formed call comes out.
    if item["category"].endswith("irrelevance"):
        return not calls
    if not calls:
        return False
    expected = item["ground_truth"]
    if len(calls) != len(expected):
        return False
    schemas = {f["name"]: f["parameters"] for f in item["functions"]}
    unused = list(range(len(calls)))
    for exp in expected:                                   # any order: give each expected call its own match
        hit = next((i for i in unused if call_ok(calls[i], exp, schemas.get(next(iter(exp)), {}))), None)
        if hit is None:
            return False
        unused.remove(hit)
    return True

official = C.official_check()                      # BFCL's ast_checker from the bfcl-eval package
for name, t, parse in [("Qwen3.5 XML", tok, parse_xml_calls), ("Qwen3 JSON", ref_tok, C.parse_json_calls)]:
    ours = theirs = agree = 0
    for it in items:
        calls = parse(C.gt_reply(t, it), it["tools"])
        a, b = check(it, calls), official(it, calls)
        ours, theirs, agree = ours + a, theirs + b, agree + (a == b)
    print(name, "| right answers scored correct, ours", ours, "BFCL", theirs, "of", len(items), "| agree", agree)

## Step 7. Score the base model on a T4

What you should see. Generation takes 299 s for the 310 items in our run. The summary shows 75.2% correct overall, 90% on simple and 58% on irrelevance, with a parse rate of 0.9935. Your check agrees with BFCL's on all 310 replies. The first reply is one sentence of plan, then a well-formed call to `calculate_triangle_area`.

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(MAIN, dtype=torch.float16).to("cuda").eval()
t0 = time.time()
base_out = C.generate(model, tok, items, batch_size=32, max_new_tokens=384)
base_sec = round(time.time() - t0)
base = C.score(items, base_out, parse_xml_calls)
print("base model |", base_sec, "s |", json.dumps(C.summary(base), indent=1))
print("agreement with BFCL's checker:", sum(official(it, r["calls"]) == r["correct"] for it, r in zip(items, base)), "of", len(items))
print(base_out[0][:300])

## Step 8. Fine-tune with LoRA and score again

What you should see. Training prints 300 steps in 26.7 min in our run, peaking at 6.84 GB, with the loss going from 0.453 to 0.012. After LoRA the summary shows 79.0% overall, 95% simple, 82% parallel and 35% irrelevance. The last line prints the gain and its interval, (0.0387, -0.0065, 0.0871).

In [ ]:
def train_lora(name, tok, rows, seed=0, rank=16, lr=2e-4, out="lora-out", special=None):
    import torch
    from datasets import Dataset
    from peft import LoraConfig, get_peft_model
    from transformers import AutoModelForCausalLM
    from trl import SFTConfig, SFTTrainer
    model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16).to("cuda")   # frozen base, fp16
    model.enable_input_require_grads()
    if special:                                          # step 10: give untrained special tokens distinct rows
        E = model.get_input_embeddings().weight
        with torch.no_grad():
            for s in special:                            # the mean of the rows of its spelling, "<" + "tool_call>"
                pieces = tok.encode(s[0], add_special_tokens=False) + tok.encode(s[1:], add_special_tokens=False)
                E[tok.convert_tokens_to_ids(s)] = E[pieces].mean(0)
    model = get_peft_model(model, LoraConfig(r=rank, lora_alpha=2 * rank, lora_dropout=0.0,
                                             target_modules="all-linear", task_type="CAUSAL_LM"))
    args = SFTConfig(output_dir=out, report_to="none", save_strategy="no", seed=seed, data_seed=seed,
                     fp16=True, bf16=False,              # a T4 has no bf16 tensor cores; SFTConfig defaults to bf16
                     learning_rate=lr, lr_scheduler_type="cosine", warmup_steps=10, num_train_epochs=1,
                     per_device_train_batch_size=2, gradient_accumulation_steps=2, max_length=1536,
                     gradient_checkpointing=True, logging_steps=5)
    trainer = SFTTrainer(model=model, args=args, processing_class=tok, train_dataset=Dataset.from_list(rows))
    for p in trainer.model.parameters():                 # GradScaler needs fp32 adapters; re-cast any that are not
        if p.requires_grad and p.dtype != torch.float32:
            p.data = p.data.float()
    trainer.train()
    return trainer

USE_OUR_ADAPTER = False        # True skips training and loads our seed-0 adapter from the Hugging Face Hub
model = None
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
if USE_OUR_ADAPTER:
    from huggingface_hub import snapshot_download
    from peft import PeftModel
    hub = snapshot_download("aimlcompanion/llm-training-lab", allow_patterns=["lltToolUse/qwen3.5-2b-lora-s0/*"])
    base_model = AutoModelForCausalLM.from_pretrained(MAIN, dtype=torch.float16).to("cuda")
    model = PeftModel.from_pretrained(base_model, os.path.join(hub, "lltToolUse", "qwen3.5-2b-lora-s0"))
else:
    t0 = time.time()
    trainer = train_lora(MAIN, tok, rows, seed=0)
    train_sec, train_peak = round(time.time() - t0), round(torch.cuda.max_memory_allocated() / 1e9, 2)
    logs = [h for h in trainer.state.log_history if "loss" in h]
    print("steps", trainer.state.global_step, "| loss", logs[0]["loss"], "->", logs[-1]["loss"], "| minutes", round(train_sec / 60, 1),
          "| peak GB", train_peak)
    model = trainer.model
    model.save_pretrained("tooluse-lora-s0")
model.eval()
t0 = time.time()
sft_out = C.generate(model, tok, items, batch_size=32, max_new_tokens=384)
sft_sec = round(time.time() - t0)
sft = C.score(items, sft_out, parse_xml_calls)
print("after LoRA |", json.dumps(C.summary(sft), indent=1))
print("gain and 95% CI:", C.paired_bootstrap([r["correct"] for r in base], [r["correct"] for r in sft]))

## Step 9. Force well-formed calls with constrained decoding

What you should see. Both constrained passes together take 619 s in our run. The table prints base constrained at 0.6645 overall with irrelevance 0.0333, and LoRA constrained at 0.6903 with irrelevance 0.0167. Simple stays at 0.93 and 0.95.

In [ ]:
t0 = time.time()
sft_forced_out = C.generate(model, tok, items, constrained=True)
with model.disable_adapter():                      # the same weights without the adapters: the base model
    base_forced_out = C.generate(model, tok, items, constrained=True)
forced_sec = round(time.time() - t0)
sft_forced = C.score(items, sft_forced_out, parse_xml_calls)
base_forced = C.score(items, base_forced_out, parse_xml_calls)
print("constrained decoding, both models |", forced_sec, "s")
for name, r in [("base", base), ("base, constrained", base_forced), ("LoRA", sft), ("LoRA, constrained", sft_forced)]:
    s = C.summary(r)
    print(name.ljust(18), "| parsed", s["parseRate"], "| correct", s["accuracy"],
          "| irrelevance", s["byCategory"]["irrelevance"]["accuracy"], "| simple", s["byCategory"]["simple_python"]["accuracy"])

## Step 10. Train the previous Qwen generation the same way

What you should see. The cell prints the median row norm 1.6223 and 0.3748 for each of the five special tokens. With the experiment on, our three Qwen3 rows scored 0 of the 250 relevant items each. The two trainings took 879 s and 982 s and peaked at 4.53 GB (lltToolUse-T2-q3-1.7b-lora-s0, -q3-1.7b-lora-rows-s0).

In [ ]:

# Kaggle validation only: record every figure the page quotes.
import json, platform, transformers, trl, peft, datasets, accelerate
def pack(rows_, outs, sec=None):
    return {"genSec": sec, "summary": C.summary(rows_), "outputs": outs, "correct": [bool(r["correct"]) for r in rows_],
            "parsed": [bool(r["parsed"]) for r in rows_], "officialCorrect": [bool(official(it, r["calls"])) for it, r in zip(items, rows_)]}
res = {"env": {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
               "transformers": transformers.__version__, "trl": trl.__version__, "peft": peft.__version__,
               "datasets": datasets.__version__, "accelerate": accelerate.__version__, "gpu": torch.cuda.get_device_name(0)},
       "check": {"labelsEqual": bool(labels_equal), "own": total / count, "trl": lib, "labelledTokens": count},
       "items": [it["id"] for it in items], "categories": [it["category"] for it in items],
       "base": pack(base, base_out, base_sec), "sft": pack(sft, sft_out, sft_sec),
       "baseForced": pack(base_forced, base_forced_out, None), "sftForced": pack(sft_forced, sft_forced_out, None), "forcedSec": forced_sec,
       "pairedBootstrap": C.paired_bootstrap([r["correct"] for r in base], [r["correct"] for r in sft])}
if not USE_OUR_ADAPTER:
    res["train"] = {"sec": train_sec, "peakGB": train_peak, "steps": trainer.state.global_step,
                    "trainable": sum(p.numel() for p in model.parameters() if p.requires_grad),
                    "loss": [{"step": h["step"], "loss": h["loss"]} for h in trainer.state.log_history if "loss" in h]}
json.dump(res, open("/kaggle/working/notebook_results.json", "w"), indent=1, ensure_ascii=False)
print({k: v for k, v in res.items() if k in ("env", "check", "pairedBootstrap", "forcedSec")})


In [ ]:
from transformers import AutoModelForCausalLM

SPECIAL = ["<tool_call>", "</tool_call>", "<|im_end|>", "<think>", "</think>"]
ref_check = AutoModelForCausalLM.from_pretrained(REF, dtype=torch.float16)      # on the CPU, to read its embedding rows
E = ref_check.get_input_embeddings().weight.float()
norms = E.norm(dim=1)
print("Qwen3-1.7B-Base | median row norm", round(norms.median().item(), 4),
      "|", {s: round(norms[ref_tok.convert_tokens_to_ids(s)].item(), 4) for s in SPECIAL})
del ref_check, E

def report(name, outs, rows_):
    relevant = [r for r in rows_ if r["category"] != "irrelevance"]
    print(name, "| relevant items right", sum(r["correct"] for r in relevant), "of", len(relevant),
          "| replies with a <tool_call> tag", sum("<tool_call>" in o for o in outs), "of", len(outs),
          "| accuracy", C.summary(rows_)["accuracy"])

RUN_EXPERIMENT = True      # True trains both Qwen3 arms on your T4; our numbers are in the table below
ref_results = {}
if RUN_EXPERIMENT:
    model = trainer = base_model = None            # drop every reference to the 2B model and its optimizer state
    gc.collect()
    torch.cuda.empty_cache()
    # TRL has no training template for Qwen3-Base, so render with its own template; the label rule is the same
    REF_HEADER = ref_tok.encode("<|im_start|>assistant" + C.NL, add_special_tokens=False)
    REF_END, REF_NL = ref_tok.convert_tokens_to_ids("<|im_end|>"), ref_tok.encode(C.NL, add_special_tokens=False)[0]
    ref_rows = []
    for c in train:
        ids = list(ref_tok.apply_chat_template(c["messages"], tools=c["tools"])["input_ids"])
        ref_rows.append({"input_ids": ids, "labels": build_labels(ids, REF_HEADER, REF_END, REF_NL)})
    ref_model = AutoModelForCausalLM.from_pretrained(REF, dtype=torch.float16).to("cuda").eval()
    outs = C.generate(ref_model, ref_tok, items)
    ref_results["base"] = (outs, C.score(items, outs, C.parse_json_calls))
    report("Qwen3-1.7B-Base", *ref_results["base"])
    del ref_model
    for name, special in [("LoRA", None), ("LoRA, special rows set first", SPECIAL)]:
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        ref_trainer = train_lora(REF, ref_tok, ref_rows, seed=0, special=special)
        outs = C.generate(ref_trainer.model.eval(), ref_tok, items)
        ref_results[name] = (outs, C.score(items, outs, C.parse_json_calls))
        report(name, *ref_results[name])
        print("   peak GB", round(torch.cuda.max_memory_allocated() / 1e9, 2))
        ref_trainer = None

## Step 11. Teach the model when not to call

What you should see. The cell prints 200 requests with no fitting tool and 1,200 conversations in the mixed set. The first one is a SQL request paired with `get_order_status`, `cancel_order`, `return_order` and `track_package`.

In [ ]:
def make_negatives(rows, start, count, shift):
    # Pair each request with the tools of another conversation that share no function name with its own tools.
    # The target reply is plain text with no call: what BFCL's irrelevance items ask for.
    out = []
    for k in range(start, start + count):
        a, b = rows[k], rows[(k - shift) % len(rows)]
        own = {t["function"]["name"] for t in a["tools"]}
        other = {t["function"]["name"] for t in b["tools"]}
        if own & other:
            continue
        out.append({"messages": [a["messages"][0], {"role": "assistant", "content": C.NO_TOOL_REPLY}], "tools": b["tools"]})
    return out

third = N_PER_SOURCE // 3
negatives = make_negatives(train, 2 * third, third, N_PER_SOURCE // 2)     # the last third of the single-turn rows
mixed = train[:2 * third] + negatives + train[N_PER_SOURCE:]
print(len(negatives), "requests with no fitting tool |", len(mixed), "conversations in the mixed set")
print(negatives[0]["messages"][0]["content"][:100], "->", [t["function"]["name"] for t in negatives[0]["tools"]])

RUN_EXPERIMENT = True      # True trains on the mixed set on your T4; our numbers are in the table below
if RUN_EXPERIMENT:
    model = trainer = ref_trainer = base_model = None
    gc.collect()
    torch.cuda.empty_cache()
    mixed_rows = []
    for c in mixed:
        ids = render(c)
        mixed_rows.append({"input_ids": ids, "labels": build_labels(ids, HEADER, IM_END, NEWLINE)})
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    mix_trainer = train_lora(MAIN, tok, mixed_rows, seed=0)
    print("trained in", round(time.time() - t0), "s | peak GB", round(torch.cuda.max_memory_allocated() / 1e9, 2))
    mix_out = C.generate(mix_trainer.model.eval(), tok, items)
    mix = C.score(items, mix_out, parse_xml_calls)
    print("mixed set + LoRA", json.dumps(C.summary(mix)))
    print("change against the base, 95% CI:", C.paired_bootstrap([r["correct"] for r in base], [r["correct"] for r in mix]))

In [ ]:

# Kaggle validation only: record steps 10 and 11 as printed above.
def pack2(outs, rows_):
    return {"summary": C.summary(rows_), "outputs": outs, "correct": [bool(r["correct"]) for r in rows_],
            "tagReplies": sum("<tool_call>" in o for o in outs)}
res2 = {"step10": {k: pack2(*v) for k, v in ref_results.items()},
        "step11": {**pack2(mix_out, mix), "pairedBootstrapVsBase": C.paired_bootstrap([r["correct"] for r in base], [r["correct"] for r in mix])}}
json.dump(res2, open("/kaggle/working/notebook_final_exp.json", "w"), indent=1, ensure_ascii=False)
print({k: v["summary"]["accuracy"] for k, v in res2["step10"].items()}, res2["step11"]["summary"]["accuracy"])
mix_trainer = None
gc.collect()
torch.cuda.empty_cache()
print("GPU memory allocated before the adapter pass, GB", round(torch.cuda.memory_allocated() / 1e9, 2))


In [ ]:
def train_lora(name, tok, rows, seed=0, rank=16, lr=2e-4, out="lora-out", special=None):
    import torch
    from datasets import Dataset
    from peft import LoraConfig, get_peft_model
    from transformers import AutoModelForCausalLM
    from trl import SFTConfig, SFTTrainer
    model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16).to("cuda")   # frozen base, fp16
    model.enable_input_require_grads()
    if special:                                          # step 10: give untrained special tokens distinct rows
        E = model.get_input_embeddings().weight
        with torch.no_grad():
            for s in special:                            # the mean of the rows of its spelling, "<" + "tool_call>"
                pieces = tok.encode(s[0], add_special_tokens=False) + tok.encode(s[1:], add_special_tokens=False)
                E[tok.convert_tokens_to_ids(s)] = E[pieces].mean(0)
    model = get_peft_model(model, LoraConfig(r=rank, lora_alpha=2 * rank, lora_dropout=0.0,
                                             target_modules="all-linear", task_type="CAUSAL_LM"))
    args = SFTConfig(output_dir=out, report_to="none", save_strategy="no", seed=seed, data_seed=seed,
                     fp16=True, bf16=False,              # a T4 has no bf16 tensor cores; SFTConfig defaults to bf16
                     learning_rate=lr, lr_scheduler_type="cosine", warmup_steps=10, num_train_epochs=1,
                     per_device_train_batch_size=2, gradient_accumulation_steps=2, max_length=1536,
                     gradient_checkpointing=True, logging_steps=5)
    trainer = SFTTrainer(model=model, args=args, processing_class=tok, train_dataset=Dataset.from_list(rows))
    for p in trainer.model.parameters():                 # GradScaler needs fp32 adapters; re-cast any that are not
        if p.requires_grad and p.dtype != torch.float32:
            p.data = p.data.float()
    trainer.train()
    return trainer

USE_OUR_ADAPTER = True        # True skips training and loads our seed-0 adapter from the Hugging Face Hub
model = None
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
if USE_OUR_ADAPTER:
    from huggingface_hub import snapshot_download
    from peft import PeftModel
    hub = snapshot_download("aimlcompanion/llm-training-lab", allow_patterns=["lltToolUse/qwen3.5-2b-lora-s0/*"])
    base_model = AutoModelForCausalLM.from_pretrained(MAIN, dtype=torch.float16).to("cuda")
    model = PeftModel.from_pretrained(base_model, os.path.join(hub, "lltToolUse", "qwen3.5-2b-lora-s0"))
else:
    t0 = time.time()
    trainer = train_lora(MAIN, tok, rows, seed=0)
    train_sec, train_peak = round(time.time() - t0), round(torch.cuda.max_memory_allocated() / 1e9, 2)
    logs = [h for h in trainer.state.log_history if "loss" in h]
    print("steps", trainer.state.global_step, "| loss", logs[0]["loss"], "->", logs[-1]["loss"], "| minutes", round(train_sec / 60, 1),
          "| peak GB", train_peak)
    model = trainer.model
    model.save_pretrained("tooluse-lora-s0")
model.eval()
t0 = time.time()
sft_out = C.generate(model, tok, items, batch_size=32, max_new_tokens=384)
sft_sec = round(time.time() - t0)
sft = C.score(items, sft_out, parse_xml_calls)
print("after LoRA |", json.dumps(C.summary(sft), indent=1))
print("gain and 95% CI:", C.paired_bootstrap([r["correct"] for r in base], [r["correct"] for r in sft]))

In [ ]:

# Kaggle validation only: the Hub adapter path of step 8.
res3 = {"summary": C.summary(sft), "outputs": sft_out, "correct": [bool(r["correct"]) for r in sft], "genSec": sft_sec,
        "pairedBootstrapVsBase": C.paired_bootstrap([r["correct"] for r in base], [r["correct"] for r in sft])}
json.dump(res3, open("/kaggle/working/notebook_final_adapter.json", "w"), indent=1, ensure_ascii=False)
print("adapter pass", res3["summary"]["accuracy"], res3["pairedBootstrapVsBase"])


## Prove it 1

Write `call_ok` and `check` in plain Python, a BFCL-style AST check for parsed tool calls. Run the cell, and it prints All checks passed when both are right. The checks cover optional parameters, string case, wrong values, missing and unknown parameters, parallel calls in any order and irrelevance items.

In [ ]:
# A BFCL-style checker for parsed tool calls. A call is {name: {param: value}}.
# An expected call is {name: {param: [allowed values]}}; "" in the list means the parameter may be left out.

def norm(s):
    for ch in " ,./-_*^":
        s = s.replace(ch, "")
    return s.lower()

def value_ok(value, allowed):
    for a in allowed:
        if isinstance(a, str) and isinstance(value, str) and norm(a) == norm(value):
            return True
        if not isinstance(a, str) and not isinstance(value, str) and a == value and type(a) is not bool and type(value) is not bool:
            return True
        if type(a) is bool and type(value) is bool and a == value:
            return True
    return False

def call_ok(call, expected, schema):
    # True when call matches expected: same name, required params present, no unknown params,
    # every left-out param allowed to be left out, every value allowed.
    name, allowed = next(iter(expected.items()))
    if name not in call:
        return False
    args = call[name]
    if any(p not in args for p in schema.get("required", [])):
        return False
    if any(p not in allowed or p not in schema.get("properties", {}) for p in args):
        return False
    if any(p not in args and "" not in allowed[p] for p in allowed):
        return False
    return all(value_ok(v, allowed[p]) for p, v in args.items())

def check(item, calls):
    # calls: a list of calls, [] for no call, None for a malformed reply.
    # Irrelevance items are right when no well-formed call comes out. Otherwise the calls must match the
    # expected list one to one, in any order.
    if item["category"] == "irrelevance":
        return not calls
    if not calls or len(calls) != len(item["expected"]):
        return False
    unused = list(range(len(calls)))
    for exp in item["expected"]:
        schema = item["schemas"][next(iter(exp))]
        hit = next((i for i in unused if call_ok(calls[i], exp, schema)), None)
        if hit is None:
            return False
        unused.remove(hit)
    return True

# Checks. Run the cell; it prints All checks passed when call_ok and check are right.
area = {"required": ["base", "height"], "properties": {"base": {}, "height": {}, "unit": {}}}
simple = {"category": "simple", "schemas": {"area": area},
          "expected": [{"area": {"base": [10], "height": [5], "unit": ["units", ""]}}]}
assert check(simple, [{"area": {"base": 10, "height": 5}}]), "unit may be left out"
assert check(simple, [{"area": {"base": 10, "height": 5, "unit": "Units"}}]), "strings compare without case"
assert not check(simple, [{"area": {"base": 10, "height": 6}}]), "wrong value"
assert not check(simple, [{"area": {"base": 10}}]), "height is required"
assert not check(simple, [{"area": {"base": 10, "height": 5, "colour": "red"}}]), "colour is not a parameter"
assert not check(simple, [{"volume": {"base": 10, "height": 5}}]), "wrong function"
assert not check(simple, []) and not check(simple, None), "no call, or a malformed one, is wrong"
power = {"required": ["base", "exponent"], "properties": {"base": {}, "exponent": {}}}
parallel = {"category": "parallel", "schemas": {"power": power},
            "expected": [{"power": {"base": [2], "exponent": [3]}}, {"power": {"base": [3], "exponent": [5]}}]}
assert check(parallel, [{"power": {"base": 3, "exponent": 5}}, {"power": {"base": 2, "exponent": 3}}]), "any order"
assert not check(parallel, [{"power": {"base": 2, "exponent": 3}}]), "one call is missing"
assert not check(parallel, [{"power": {"base": 2, "exponent": 3}}, {"power": {"base": 2, "exponent": 3}}]), "one call may match one expected call only"
irrelevant = {"category": "irrelevance", "schemas": {"area": area}, "expected": None}
assert check(irrelevant, []) and check(irrelevant, None), "no well-formed call is right here"
assert not check(irrelevant, [{"area": {"base": 1, "height": 1}}]), "any call is wrong here"
assert not call_ok({"area": {"base": True, "height": 5}}, simple["expected"][0], area), "True is not 10"
print("All checks passed")